# Notebook 14 – Domain-Based Feature Engineering

## 1. Domain Selection: E-commerce

Domain-based feature engineering means using knowledge of a specific business domain to create meaningful features from raw data.

For this notebook, we use the **E-commerce** domain.

The goal is to create features that represent customer behavior, purchasing patterns, order value, and product performance.

Examples include:
- Customer spending
- Purchase frequency
- Average order value
- Discount percentage
- Product popularity
- Customer recency

The main focus is not only creating new columns, but understanding **why the feature is useful for the business and Machine Learning model**.

In [1]:
import pandas as pd

data = pd.DataFrame({
    "CustomerID": [101, 101, 102, 103, 103],
    "OrderAmount": [1200, 800, 500, 1500, 700],
    "Quantity": [3, 2, 1, 4, 2],
    "ProductPrice": [400, 400, 500, 375, 350],
    "Discount": [100, 50, 0, 150, 70],
    "DaysSinceLastPurchase": [10, 10, 30, 5, 5],
    "ProductRating": [4.5, 4.0, 3.5, 4.8, 4.2]
})

data.head()

,CustomerID,OrderAmount,Quantity,ProductPrice,Discount,DaysSinceLastPurchase,ProductRating
0,101,1200,3,400,100,10,4.5
1,101,800,2,400,50,10,4.0
2,102,500,1,500,0,30,3.5
3,103,1500,4,375,150,5,4.8
4,103,700,2,350,70,5,4.2


## 2. Why Domain Knowledge is Important

Domain knowledge helps us create features that have real business meaning.

In E-commerce, simply using raw order amounts may not fully represent customer behavior.

For example:
- Total spending can represent customer value.
- Purchase frequency can represent customer engagement.
- Average order value can represent spending behavior.
- Discount percentage can represent price sensitivity.

Good domain-based features should be meaningful, useful for prediction, and available at the time of prediction.

In [2]:
print("Domain selected: E-commerce")
print("Raw features:", list(data.columns))

Domain selected: E-commerce
Raw features: ['CustomerID', 'OrderAmount', 'Quantity', 'ProductPrice', 'Discount', 'DaysSinceLastPurchase', 'ProductRating']


## 3. Feature 1 – Total Customer Spending

### Feature Name
Total_Spending

### Source Columns
CustomerID, OrderAmount

### Logic
Sum of all order amounts for each customer.

### Reason
To measure the total amount spent by a customer.

### Business Meaning
Represents the customer's overall spending value.

### ML Relevance
Can help identify high-value customers and predict customer behavior.

### Leakage Check
The aggregation must use only information available before the prediction point.

### Final Decision
Retain

In [3]:
data["Total_Spending"] = data.groupby("CustomerID")["OrderAmount"].transform("sum")

print(data[["CustomerID", "OrderAmount", "Total_Spending"]])

   CustomerID  OrderAmount  Total_Spending
0         101         1200            2000
1         101          800            2000
2         102          500             500
3         103         1500            2200
4         103          700            2200


## 4. Feature 2 – Purchase Frequency

### Feature Name
Purchase_Frequency

### Source Columns
CustomerID

### Logic
Count the number of orders made by each customer.

### Reason
To measure how frequently a customer purchases.

### Business Meaning
Represents customer engagement and purchasing activity.

### ML Relevance
Can help predict repeat purchases, churn, or customer value.

### Leakage Check
Count only purchases that occurred before the prediction time.

### Final Decision
Retain

In [4]:
data["Purchase_Frequency"] = data.groupby("CustomerID")["CustomerID"].transform("count")

print(data[["CustomerID", "Purchase_Frequency"]].drop_duplicates())

   CustomerID  Purchase_Frequency
0         101                   2
2         102                   1
3         103                   2


## 5. Feature 3 – Average Order Value

### Feature Name
Average_Order_Value

### Source Columns
CustomerID, OrderAmount

### Logic
Total customer spending divided by the number of orders.

### Reason
To understand the average amount spent per order.

### Business Meaning
Represents the typical order value of a customer.

### ML Relevance
Can help identify high-value purchasing behavior.

### Leakage Check
Use only historical orders available before prediction.

### Final Decision
Retain

In [5]:
data["Average_Order_Value"] = (
    data["Total_Spending"] / data["Purchase_Frequency"]
)

print(data[["CustomerID", "Average_Order_Value"]].drop_duplicates())

   CustomerID  Average_Order_Value
0         101               1000.0
2         102                500.0
3         103               1100.0


## 6. Feature 4 – Discount Percentage

### Feature Name
Discount_Percentage

### Source Columns
OrderAmount, Discount

### Logic
Discount divided by order amount, multiplied by 100.

### Reason
To measure how much discount was applied to an order.

### Business Meaning
Represents the customer's exposure to discounts.

### ML Relevance
Can help identify price-sensitive customers and purchasing patterns.

### Leakage Check
The discount must be known before or at the time of purchase prediction.

### Final Decision
Retain

In [6]:
data["Discount_Percentage"] = (
    data["Discount"] / data["OrderAmount"] * 100
)

print(data[["OrderAmount", "Discount", "Discount_Percentage"]])

   OrderAmount  Discount  Discount_Percentage
0         1200       100             8.333333
1          800        50             6.250000
2          500         0             0.000000
3         1500       150            10.000000
4          700        70            10.000000


## 7. Feature 5 – Average Quantity per Order

### Feature Name
Average_Quantity

### Source Columns
CustomerID, Quantity

### Logic
Calculate the average quantity purchased by each customer.

### Reason
To understand the customer's typical purchase size.

### Business Meaning
Represents the average number of items purchased per order.

### ML Relevance
Can help identify bulk-buying or regular purchasing behavior.

### Leakage Check
Use only previous orders when making a future prediction.

### Final Decision
Retain

In [7]:
data["Average_Quantity"] = data.groupby("CustomerID")["Quantity"].transform("mean")

print(data[["CustomerID", "Average_Quantity"]].drop_duplicates())

   CustomerID  Average_Quantity
0         101               2.5
2         102               1.0
3         103               3.0


## 8. Feature 6 – Customer Recency

### Feature Name
Customer_Recency

### Source Columns
DaysSinceLastPurchase

### Logic
Use the number of days since the customer's last purchase.

### Reason
To understand how recently the customer purchased.

### Business Meaning
A lower value means the customer purchased more recently.

### ML Relevance
Recency is useful for predicting customer engagement and churn.

### Leakage Check
The value must be calculated using purchases available before the prediction date.

### Final Decision
Retain

In [8]:
data["Customer_Recency"] = data["DaysSinceLastPurchase"]

print(data[["CustomerID", "Customer_Recency"]].drop_duplicates())

   CustomerID  Customer_Recency
0         101                10
2         102                30
3         103                 5


## 9. Feature 7 – Total Items Purchased

### Feature Name
Total_Items_Purchased

### Source Columns
CustomerID, Quantity

### Logic
Sum the quantity of items purchased by each customer.

### Reason
To measure the total number of products purchased.

### Business Meaning
Represents the customer's overall purchase volume.

### ML Relevance
Can help identify active and high-volume customers.

### Leakage Check
Use only historical purchases available before prediction.

### Final Decision
Retain

In [9]:
data["Total_Items_Purchased"] = data.groupby("CustomerID")["Quantity"].transform("sum")

print(data[["CustomerID", "Total_Items_Purchased"]].drop_duplicates())

   CustomerID  Total_Items_Purchased
0         101                      5
2         102                      1
3         103                      6


## 10. Feature 8 – Average Product Rating

### Feature Name
Average_Product_Rating

### Source Columns
CustomerID, ProductRating

### Logic
Calculate the average product rating associated with a customer's purchases.

### Reason
To understand the quality or preference of products purchased.

### Business Meaning
Represents the average rating of purchased products.

### ML Relevance
Can help understand customer preferences and predict future purchases.

### Leakage Check
Use only ratings that were available before the prediction point.

### Final Decision
Needs Further Analysis

In [10]:
data["Average_Product_Rating"] = (
    data.groupby("CustomerID")["ProductRating"].transform("mean")
)

print(data[["CustomerID", "Average_Product_Rating"]].drop_duplicates())

   CustomerID  Average_Product_Rating
0         101                    4.25
2         102                    3.50
3         103                    4.50


## 11. Feature 9 – Average Price per Item

### Feature Name
Average_Price_Per_Item

### Source Columns
OrderAmount, Quantity

### Logic
OrderAmount divided by Quantity.

### Reason
To estimate the average amount spent per item.

### Business Meaning
Represents the average price paid for each item in an order.

### ML Relevance
Can help identify premium or low-cost purchasing behavior.

### Leakage Check
The order amount and quantity must be available at prediction time.

### Final Decision
Retain

In [11]:
data["Average_Price_Per_Item"] = (
    data["OrderAmount"] / data["Quantity"]
)

print(data[["OrderAmount", "Quantity", "Average_Price_Per_Item"]])

   OrderAmount  Quantity  Average_Price_Per_Item
0         1200         3                   400.0
1          800         2                   400.0
2          500         1                   500.0
3         1500         4                   375.0
4          700         2                   350.0


## 12. Feature 10 – Customer Value Score

### Feature Name
Customer_Value_Score

### Source Columns
Total_Spending, Purchase_Frequency

### Logic
Total spending divided by purchase frequency.

### Reason
To create a simple measure of customer purchasing value.

### Business Meaning
Represents the average spending level per purchase.

### ML Relevance
Can help segment customers into different value groups.

### Leakage Check
Use only historical customer transactions available before prediction.

### Final Decision
Needs Further Analysis

In [12]:
data["Customer_Value_Score"] = (
    data["Total_Spending"] / data["Purchase_Frequency"]
)

print(data[["CustomerID", "Customer_Value_Score"]].drop_duplicates())

   CustomerID  Customer_Value_Score
0         101                1000.0
2         102                 500.0
3         103                1100.0


## 13. Feature Engineering Summary

The following features were created using E-commerce domain knowledge:

1. Total_Spending
2. Purchase_Frequency
3. Average_Order_Value
4. Discount_Percentage
5. Average_Quantity
6. Customer_Recency
7. Total_Items_Purchased
8. Average_Product_Rating
9. Average_Price_Per_Item
10. Customer_Value_Score

These features represent customer value, purchase behavior, product interaction, pricing, and engagement.

The final decision for each feature should consider business usefulness, ML performance, redundancy, and leakage risk.

In [13]:
summary = pd.DataFrame([
    ["Total_Spending", "CustomerID, OrderAmount", "Sum orders", "Retain"],
    ["Purchase_Frequency", "CustomerID", "Count orders", "Retain"],
    ["Average_Order_Value", "CustomerID, OrderAmount", "Total / orders", "Retain"],
    ["Discount_Percentage", "OrderAmount, Discount", "Discount / Order × 100", "Retain"],
    ["Average_Quantity", "CustomerID, Quantity", "Mean quantity", "Retain"],
    ["Customer_Recency", "DaysSinceLastPurchase", "Days since purchase", "Retain"],
    ["Total_Items_Purchased", "CustomerID, Quantity", "Sum quantity", "Retain"],
    ["Average_Product_Rating", "CustomerID, ProductRating", "Mean rating", "Needs Further Analysis"],
    ["Average_Price_Per_Item", "OrderAmount, Quantity", "Order / quantity", "Retain"],
    ["Customer_Value_Score", "Total_Spending, Purchase_Frequency", "Total / frequency", "Needs Further Analysis"]
], columns=[
    "Feature Name", "Source Columns", "Logic", "Final Decision"
])

summary

,Feature Name,Source Columns,Logic,Final Decision
0,Total_Spending,"CustomerID, OrderAmount",Sum orders,Retain
1,Purchase_Frequency,CustomerID,Count orders,Retain
2,Average_Order_Value,"CustomerID, OrderAmount",Total / orders,Retain
3,Discount_Percentage,"OrderAmount, Discount",Discount / Order × 100,Retain
4,Average_Quantity,"CustomerID, Quantity",Mean quantity,Retain
5,Customer_Recency,DaysSinceLastPurchase,Days since purchase,Retain
6,Total_Items_Purchased,"CustomerID, Quantity",Sum quantity,Retain
7,Average_Product_Rating,"CustomerID, ProductRating",Mean rating,Needs Further Analysis
8,Average_Price_Per_Item,"OrderAmount, Quantity",Order / quantity,Retain
9,Customer_Value_Score,"Total_Spending, Purchase_Frequency",Total / frequency,Needs Further Analysis


## 14. Required Documentation

For every important engineered feature, we document:

### Feature Name
Name of the newly created feature.

### Source Columns
Columns used to create the feature.

### Logic
Formula or business rule used to create the feature.

### Reason
Why the feature was created.

### Business Meaning
What the feature represents in the E-commerce business.

### ML Relevance
How the feature could help a Machine Learning model.

### Leakage Check
Whether the feature could use information that would not be available at prediction time.

### Final Decision
The feature is classified as:
- Retain
- Remove
- Needs Further Analysis

In [14]:
print("Total engineered features:", len(summary))
print("\nFeature engineering documentation completed.")
print(summary["Final Decision"].value_counts())

Total engineered features: 10

Feature engineering documentation completed.
Final Decision
Retain                    8
Needs Further Analysis    2
Name: count, dtype: int64


## 15. Conclusion

Domain-based feature engineering uses business knowledge to create meaningful features from raw data.

In this notebook, the E-commerce domain was used to create 10 features related to customer spending, purchase frequency, order value, discounts, quantity, recency, and product behavior.

The key takeaway is:

**Good feature engineering connects raw data with real business meaning and helps the Machine Learning model learn useful patterns.**

Every feature should also be checked for leakage before being used in a Machine Learning model.

In [15]:
print("Domain-Based Feature Engineering completed successfully.")
print("Domain: E-commerce")
print("Features created:", len(summary))

Domain-Based Feature Engineering completed successfully.
Domain: E-commerce
Features created: 10
